# Scraping with Web API

- [slide](https://docs.google.com/presentation/d/1gn4d5gzXzgyEAIz_AOdM3pmJ0-6iiN7EbTjKqCQlgPo/edit?usp=sharing) for getting json data online
- Finding data url
- Scraping 104.com job info
- Understanding how to send request and get back response
- Send request with Cookie, payload, Referer...

## Using requests library to send web requests

* Document: http://docs.python-requests.org/en/master/ 
* Quickstart http://docs.python-requests.org/en/master/user/quickstart/

In [1]:
import requests
import json
response = requests.get('https://tcgbusfs.blob.core.windows.net/blobyoubike/YouBikeTP.gz')
response.json()
type(response.json())

dict

### Practice: Find out and traverse data behind urls

逐一測試過每一個link是否都讀得到資料，以獲得對這些資料概略了解。
* (hint) Using `requests.get(url, timeout=(x, y))` to set the limitation of waiting time
* `timeout=(x, y)`: Max x seconds to connect to server and max y seconds to wait on response
* https://data.moi.gov.tw/moiod/System/Principle.aspx?Sample=2

#### Canyes

In [39]:
url_cnyes = "https://news.cnyes.com/api/v3/news/category/headline?startAt=1588262400&endAt=1589212799&limit=30"
res = requests.get(url_cnyes).json()
print(type(res))
print(res.keys())

<class 'dict'>
dict_keys(['items', 'message', 'statusCode'])


#### PChome 24h product search

In [35]:
url_pchome = "https://ecshweb.pchome.com.tw/search/v3.3/all/results?q=iphone&page=1&sort=sale/dc"
res = requests.get(url_pchome).json()
print(type(res))
print(res.keys())

<class 'dict'>
dict_keys(['QTime', 'totalRows', 'totalPage', 'range', 'cateName', 'q', 'subq', 'token', 'isMust', 'prods'])


#### Dcard

Dcard API最近又做了改版，用以下的Code應該沒辦法順利撈回，但你也可以測試看看，可以看到什麼樣的錯誤訊息。

In [46]:
url_dcard = "https://www.dcard.tw/service/api/v2/forums/relationship/posts?limit=50"
user_agent_dcard = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/106.0.0.0 Safari/537.36 Edg/106.0.1370.34"
headers = {'User-Agent': user_agent_dcard}
res = requests.get(url_dcard, headers = headers)
print(res)

### (Option) Write a function to load json data

In [49]:
def get_web_json(url, headers=""):
    response = requests.get(url, timeout=(3, 5), headers=headers)
    print("Response Code:", response.status_code)
    if not response.ok:
        return None
    data = response.json()
    return data

# get_web_json(url_pchome)
# get_web_json(url_cnyes)

## Scraping 104.com
* Slide: https://docs.google.com/presentation/d/e/2PACX-1vRW84XoB5sFRT1Eg-GrK4smX23qoNkFffz_h8oRU4AIvJAgrrxBn8059_0UeHv_pFBks_Z37vNbLGai/pub?start=false&loop=false&delayms=3000
```
https://www.104.com.tw/jobs/search/list?ro=0&kwop=7&keyword=data%20scientist&expansionType=area%2Cspec%2Ccom%2Cjob%2Cwf%2Cwktm&order=14&asc=0&page=2&mode=s&jobsource=2018indexpoc
```

### 1. Get the first page, but fails

* 嘗試獲取104.com的搜尋結果的第一頁資料網址，結果應該會傳回來一個空的HTML無法傳回資料。這是因為通常服務提供方會嘗試要求提出拜訪要求的瀏覽器需要提供一些簡單的驗證機制，例如是用什麼瀏覽器連上去的（Dcard要求這樣的資訊）、是從哪一個頁面跳過去的（104.com要求）。
* 撰寫爬蟲時必須「模仿」瀏覽器的機制，如果對方希望提供瀏覽器提供這樣的資訊，那撰寫爬蟲時也就要提供這樣的資訊。


In [8]:
url_104 = 'https://www.104.com.tw/jobs/search/list?ro=0&kwop=7&keyword=data%20scientist&expansionType=area%2Cspec%2Ccom%2Cjob%2Cwf%2Cwktm&order=14&asc=0&page=2&mode=s&jobsource=2018indexpoc'
response = requests.get(url_104)
print(response.status_code)
print(response.text)
print(response.headers)

200
<html xmlns="http://www.w3.org/1999/xhtml"><head><META HTTP-EQUIV="CONTENT-TYPE" CONTENT="TEXT/HTML; CHARSET=utf-8"/><title></title></head>
<body>
<SCRIPT LANGUAGE="JavaScript">
window.location="https://www.104.com.tw/jobs/main/syserr?eid=6129352555614771649";
</script>
</body>
</html>
{'Cache-Control': 'no-cache', 'Connection': 'close', 'Pragma': 'no-cache', 'Content-Type': 'text/html; charset=utf-8', 'Content-Length': '292'}


#### (Option) Write html to file

當明明知道他是一個json，卻一直無法順利用`.json()`將其拆且為`list` or `dict`的型態時，最有可能的問題是，該網址設了一些檢核機制讓我們不能那麼粗暴草率地拿到資料，導致它傳回來給我們的是一個用html編寫成的錯誤訊息，例如stuatus code: 403。這時候我要怎麼知道發生錯誤？一種方式是把status code印出來看看；另一種方式是，他可能也是傳給你status code:200，但實際上就是傳回一個告知你不能存取的html。這時候我們可以採取的作法就是把該HTML，也就是回傳的結果寫入到.html檔，然後用瀏覽器開啟看看他究竟傳回來什麼錯誤訊息。

In [10]:
with open('temp_output.html', 'w') as fout:
    fout.write(response.text)

# webbrowser cannot work, but why?
import webbrowser
webbrowser.open_new_tab('temp_output.html')

True

#### (Practice) Observing youbike data headers

Is is differnt from 104.com's?觀察看看，是否youbike也有類似的問題，或者youbike data headers是什麼？務必觀察看看，因為如果我們要很會寫爬蟲，一定要會觀察header，而這只是最簡單的一種情形。


In [11]:
import requests
import json
response = requests.get('https://tcgbusfs.blob.core.windows.net/blobyoubike/YouBikeTP.gz')
print(response)
print(response.status_code)
print(type(response)) # <class 'requests.models.Response'>
print(type(response.text)) # <class 'str'>

<Response [200]>
200
<class 'requests.models.Response'>
<class 'str'>


In [12]:
print(response.headers)
import pandas as pd
print(pd.DataFrame.from_dict(response.headers, orient='index'))

{'Content-Length': '30296', 'Content-Type': 'application/octet-stream', 'Content-Encoding': 'gzip', 'Content-MD5': 'IxgyrnXCzuzcMOhLCJ6i7A==', 'Last-Modified': 'Mon, 10 Oct 2022 04:03:54 GMT', 'ETag': '0x8DAAA747313ADF4', 'Server': 'Windows-Azure-Blob/1.0 Microsoft-HTTPAPI/2.0', 'x-ms-request-id': '5cffa851-001e-003b-735d-dcb88d000000', 'x-ms-version': '2009-09-19', 'x-ms-lease-status': 'unlocked', 'x-ms-blob-type': 'BlockBlob', 'Access-Control-Allow-Origin': '*', 'Date': 'Mon, 10 Oct 2022 04:04:05 GMT'}
                                                                        0
Content-Length                                                      30296
Content-Type                                     application/octet-stream
Content-Encoding                                                     gzip
Content-MD5                                      IxgyrnXCzuzcMOhLCJ6i7A==
Last-Modified                               Mon, 10 Oct 2022 04:03:54 GMT
ETag                                          

### 2. Add referer to get back 104 data

現在我們來加入一些對方伺服器可能會要求我們發出request的時候提供的東西，最常見的有以下的資訊。但，實際上我們得一個一個測試看看，才會知道對方要求什麼。通常會從User-agent測起，再來Referer、再來Cookie。但有經驗的人多半一看就猜得到。
* User-Agent: 你用什麼瀏覽器或系統
* Referer: 你從哪個頁面點選、跳轉過來
* Cookies: 經過與伺服器建立連結後，他給了你什麼資訊好讓你持續可以待在這個頁面。

In [13]:
url_104 = 'https://www.104.com.tw/jobs/search/list?ro=0&kwop=7&keyword=data%20scientist&expansionType=area%2Cspec%2Ccom%2Cjob%2Cwf%2Cwktm&order=14&asc=0&page=2&mode=s&jobsource=2018indexpoc'
headers = {'referer': 'https://www.104.com.tw/'}
raw = requests.get(url_104, headers=headers).json()
print(type(raw))

<class 'dict'>


### 3. Traverse data to get the data block

觀察一下資料的概況，並把他轉為pandas來觀察它。

In [50]:
print(raw.keys())
print(type(raw['data']))
print(raw['data'].keys())
print(type(raw['data']['list']))
print(type(raw['data']['list'][0]))
df = pd.DataFrame(raw['data']['list'])
df

dict_keys(['status', 'action', 'data', 'statusMsg', 'errorMsg'])
<class 'dict'>
dict_keys(['query', 'filterDesc', 'queryDesc', 'list', 'count', 'pageNo', 'totalPage', 'totalCount'])
<class 'list'>
<class 'dict'>


,jobType,jobNo,jobName,jobNameSnippet,jobRole,jobRo,jobAddrNo,jobAddrNoDesc,jobAddress,description,...,tags,landmark,link,jobsource,jobNameRaw,custNameRaw,lon,lat,remoteWorkType,major
0,0,7120552,Analytical Research - Assistant Scientist,Analytical Research - Assistant <em class='b-t...,1,1,6001001010,台北市內湖區,港墘路221巷41號3樓,.\n5.\tReport experimental [[[data]]] and stud...,...,[員工288人],,{'applyAnalyze': '//www.104.com.tw/jobs/apply/...,jolist_c_relevance,Analytical Research - Assistant Scientist,安成國際藥業股份有限公司,121.575066,25.073247,0,"[化學工程相關, 化學相關, 藥學相關]"
1,0,12712050,Data Analyst (Product),<em class='b-txt--highlight'>Data</em> Analyst...,1,1,6001001004,台北市松山區,民生東路三段156號10樓,\n\nPosition Overview\n\n[[[Data]]] Analyst (P...,...,[員工120人],,{'applyAnalyze': '//www.104.com.tw/jobs/apply/...,jolist_c_relevance,Data Analyst (Product),Vyond_香港商高創動訊有限公司台灣分公司,121.5477934,25.057203,0,[數學及電算機科學學科類]
2,0,11407165,【產品處】AI工程師/資料科學家/機器學習工程師 (可遠端作業),【產品處】AI工程師/<em class='b-txt--highlight'>資料科學家<...,1,1,6001001004,台北市松山區,松山區南京東三段285號10樓 (安泰大樓),【熱血團隊】\n域動[[[Data]]] Team是由一群對『數據分析』及『AI應用』具有研...,...,"[員工85人, 遠端工作]",距捷運南京復興站190公尺,{'applyAnalyze': '//www.104.com.tw/jobs/apply/...,jolist_c_relevance,【產品處】AI工程師/資料科學家/機器學習工程師 (可遠端作業),域動行銷股份有限公司,121.5459245,25.0520532,1,"[數學及電算機科學學科類, 電機電子工程相關, 統計學相關]"
3,0,11886237,Field Application Scientist,Field Application <em class='b-txt--highlight'...,1,1,6001001007,台北市信義區,,職務內容:\n(1)執行細胞培養及細胞分析實驗。\n(2)能獨立或合作執行多個內部或外部合作...,...,"[外商公司, 員工25人]",,{'applyAnalyze': '//www.104.com.tw/jobs/apply/...,jolist_c_relevance,Field Application Scientist,生德奈生物科技股份有限公司,121.5720055,25.0409201,0,"[生物學相關, 其他醫藥衛生相關]"
4,0,7023012,Machine Learning Scientist,Machine Learning <em class='b-txt--highlight'>...,1,1,6001001007,台北市信義區,松仁路123號6樓,analyzing and extracting valuable information...,...,[員工500人],距捷運象山站190公尺,{'applyAnalyze': '//www.104.com.tw/jobs/apply/...,jolist_c_relevance,Machine Learning Scientist,沛星互動科技股份有限公司,121.5692094,25.0342822,0,"[數學及電算機科學學科類, 應用數學相關, 資訊工程相關]"
5,2,12953849,4H新光金控_ 數據科學家,4H新光金控_ <em class='b-txt--highlight'>數據科學家</em>,1,1,6001001001,台北市中正區,忠孝西路一段66號39樓,1.掌握金融轉型趨勢，與業務單位合作進行商機探索與分析，推動集團轉型計畫與客群經營策略。\n...,...,"[上市上櫃, 員工20000人]",距捷運台北車站270公尺,{'applyAnalyze': '//www.104.com.tw/jobs/apply/...,jolist_c_relevance,4H新光金控_ 數據科學家,新光金融控股股份有限公司(新光金控/新光人壽/新光銀行/新光投信),121.5153112,25.0459441,0,"[資訊管理相關, 資訊工程相關, 數理統計相關]"
6,2,12378263,營運功能單位 軟體研發中心 數據科學家/ AI演算法工程師(台南/新竹),營運功能單位 軟體研發中心 <em class='b-txt--highlight'>數據科...,1,1,6001006001,新竹市,,1.\t數據分析\n2.\t跨部門溝通及合作\n3. 研究與開發機器學習演算法(含深...,...,"[上市上櫃, 員工43000人]",,{'applyAnalyze': '//www.104.com.tw/jobs/apply/...,jolist_c_relevance,營運功能單位 軟體研發中心 數據科學家/ AI演算法工程師(台南/新竹),光寶科技股份有限公司,120.9674798,24.8138287,0,"[數理統計相關, 資訊工程相關, 電機電子工程相關]"
7,0,12935117,R&amp;D Functional Assessment– Scientist (In v...,R&amp;D Functional Assessment– <em class='b-tx...,1,1,6001001010,台北市內湖區,,"We are seeking an exceptional, highly motivate...",...,[員工50人],,{'applyAnalyze': '//www.104.com.tw/jobs/apply/...,jolist_c_relevance,R&D Functional Assessment– Scientist (In vitro...,臺灣百濟神州有限公司,121.5909027,25.0689422,0,"[自然科學學科類, 醫藥衛生學科類, 農林漁牧學科類]"
8,0,11915941,NLP AI Scientist,NLP AI <em class='b-txt--highlight'>Scientist<...,1,1,6001001001,台北市中正區,羅斯福路二段100號25樓(古亭捷運9號出口正門),喜愛鑽研先進 Machine Learning 及 NLP 相關演算法，並具備實作能力，開發...,...,[遠端工作],距捷運古亭站60公尺,{'applyAnalyze': '//www.104.com.tw/jobs/apply/...,jolist_c_relevance,NLP AI Scientist,歐易亞科技股份有限公司,121.522215,25.0267242,2,[]
9,2,12259375,【數據部】Data Analysts/Scientist 數據分析師/科學家,【數據部】<em class='b-txt--highlight'>Data</em> An...,1,1,6001008008,台中市南屯區,大墩南路379號,"【THE PROJECT / CONTEXT】\n[[[Data]]] is fuel, t...",...,"[外商公司, 員工1000人]",距捷運豐樂公園站320公尺,{'applyAnalyze': '//www.104.com.tw/jobs/apply/...,jolist_c_relevance,【數據部】Data Analysts/Scientist 數據分析師/科學家,DECATHLON法商迪卡儂_台灣迪卡儂有限公司,120.6496192,24.1313963,0,"[資訊管理相關, 統計學相關, 其他商業及管理相關]"


### 4. Get next page: get the 2nd, 1st, 3rd, ..., page urls

接下來要從Chrome Development Tools來觀察，下二頁、三頁、四頁的網址為何（例如以下網址）。然後要去觀察這些網址的變化，應該不難觀察在page=1, page=2, page=3的數字上有所變化。通常這種網址的變化都是有規律性的。
```
https://www.104.com.tw/jobs/search/list?ro=0&kwop=7&keyword=data%20scientist&expansionType=area%2Cspec%2Ccom%2Cjob%2Cwf%2Cwktm&order=14&asc=0&page=2&mode=s&jobsource=2018indexpoc

https://www.104.com.tw/jobs/search/list?ro=0&kwop=7&keyword=data%20scientist&expansionType=area%2Cspec%2Ccom%2Cjob%2Cwf%2Cwktm&order=14&asc=0&page=1&mode=s&jobsource=2018indexpoc

https://www.104.com.tw/jobs/search/list?ro=0&kwop=7&keyword=data%20scientist&expansionType=area%2Cspec%2Ccom%2Cjob%2Cwf%2Cwktm&order=14&asc=0&page=3&mode=s&jobsource=2018indexpoc
```

In [15]:
for page in range(1, 6):
    url = 'https://www.104.com.tw/jobs/search/list?ro=0&kwop=7&keyword=data%20scientist&expansionType=area%2Cspec%2Ccom%2Cjob%2Cwf%2Cwktm&order=14&asc=0&page=' + str(page) + '&mode=s&jobsource=2018indexpoc'
    headers = {'referer': 'https://www.104.com.tw/'}
    raw = requests.get(url, headers=headers).json()
    print(len(raw['data']['list']))

30
20
20
20
20


除了列印出來以外，用`all_data`這個變項儲存資料，並在for-loop中把每一頁的資料用`extend()`附加到`all_data`中。

In [16]:
all_data = []
for page in range(1, 6):
    url = 'https://www.104.com.tw/jobs/search/list?ro=0&kwop=7&keyword=data%20scientist&expansionType=area%2Cspec%2Ccom%2Cjob%2Cwf%2Cwktm&order=14&asc=0&page=' + str(page) + '&mode=s&jobsource=2018indexpoc'
    headers = {'referer': 'https://www.104.com.tw/'}
    raw = requests.get(url, headers=headers).json()
    all_data.extend(raw['data']['list'])
    print(len(all_data))

30
50
70
90
110


最後我們把整筆資料轉為pandas比較好觀察。

In [17]:
df = pd.DataFrame(all_data)
print(df.shape)
print(len(set(df.jobNo)))

(110, 40)
110


### 5. detect ending condition

前面的步驟中我們知道要如何一步一步抓取每一頁的資料，最後還有一個問題就是，抓到什麼時候才要停？通常程式設計師也得留一個線索，才知道要如何在網頁上自動化呈現最後一頁（如同你看網頁時所看到的最後一頁的頁碼）。所以我們得去揣測程式設計師的邏輯，看他是怎麼設計最後一頁的停止機制的。

通常最主要有兩種：
1. 直接顯示最後一頁是多少：那就寫一個爬蟲，直接去偵測這個最後一頁是多少，然後把他當成ending condition。如104.com的例子是有總資料筆數的，除以頁數就可以知道總頁數。
2. 不直接顯示最後一頁是多少（e.g. PCHOME），就設一個夠大的迴圈，讓爬蟲抓抓抓抓到當掉，或者抓到資料沒再新增了，我們就偵測如果抓到的資料是零筆，就讓他跳出迴圈。

In [18]:
print(raw.keys())
print(type(raw['data']))
print(raw['data'].keys())
print(raw['data']['pageNo'])
print(raw['data']['totalPage'])
print(raw['data']['count'])
print(raw['data']['totalCount'])

dict_keys(['status', 'action', 'data', 'statusMsg', 'errorMsg'])
<class 'dict'>
dict_keys(['query', 'filterDesc', 'queryDesc', 'list', 'count', 'pageNo', 'totalPage', 'totalCount'])
5
150
['7926', '7638', '146', '29', '113', '0', '0']
7926


In [19]:
totalPage = raw['data']['totalPage']
for p in range(1, totalPage):
    url = 'https://www.104.com.tw/jobs/search/list?ro=0&kwop=7&keyword=data%20scientist&expansionType=area%2Cspec%2Ccom%2Cjob%2Cwf%2Cwktm&order=14&asc=0&page=' + str(page) + '&mode=s&jobsource=2018indexpoc'
    headers = {'referer': 'https://www.104.com.tw/'}
    raw = requests.get(url, headers=headers).json()
    all_data.extend(raw['data']['list'])
    print(len(all_data))

130
150
170
190
210
230
250
270
290
310
330
350
370
390
410
430
450
470
490
510
530
550
570
590
610
630
650
670
690
710
730
750
770
790
810
830
850
870
890
910
930
950
970
990
1010
1030
1050
1070
1090
1110
1130
1150
1170
1190
1210
1230
1250


KeyboardInterrupt: 

### 6. convert to dataframe

In [22]:
df = pd.DataFrame(all_data)
print(df.shape)
df

(1250, 40)


,jobType,jobNo,jobName,jobNameSnippet,jobRole,jobRo,jobAddrNo,jobAddrNoDesc,jobAddress,description,...,tags,landmark,link,jobsource,jobNameRaw,custNameRaw,lon,lat,remoteWorkType,major
0,1,12901787,日班倉管員(週休二日免排班),日班倉管員(週休二日免排班),1,1,6001005012,桃園市大園區,,"1.倉儲貨品管理及貨品環境維護 \n2.物流倉儲各項改包進貨及出貨作業,熟悉倉庫生態佳 \n...",...,"[外商公司, 員工650人]",,{'applyAnalyze': '//www.104.com.tw/jobs/apply/...,hotjob_chr,日班倉管員(週休二日免排班),香港商台灣利豐物流有限公司台灣分公司_LF LOGISTICS (TAIWAN) LIMITED,121.193945,25.0492632,0,[]
1,1,11726918,水晶軒專櫃店長(台中區)底薪固定13個月+獎金,水晶軒專櫃店長(台中區)底薪固定13個月+獎金,1,1,6001008004,台中市西區,,百貨公司專櫃儲備幹部及店長職務\r\n1.負責介紹及銷售商品。 \r\n2.提供顧客之接待與...,...,"[外商公司, 員工190人]",,{'applyAnalyze': '//www.104.com.tw/jobs/apply/...,hotjob_chr,水晶軒專櫃店長(台中區)底薪固定13個月+獎金,香港商施華洛世奇有限公司台灣分公司,120.6631289,24.1430604,0,[]
2,1,8422468,(Human Resources) Learning and Development Tea...,(Human Resources) Learning and Development Tea...,1,1,6001002011,新北市新店區,,About HTC\n\nHTC built a vision of the future ...,...,"[上市上櫃, 員工2000人]",,{'applyAnalyze': '//www.104.com.tw/jobs/apply/...,hotjob_chr,(Human Resources) Learning and Development Tea...,宏達電 HTC Corporation_宏達國際電子股份有限公司,121.5394822,24.978282,0,"[心理學相關, 人力資源相關]"
3,1,12388507,[2022科技菁英計畫] 產品測試工程師 Product/Test Engineer [限...,[2022科技菁英計畫] 產品測試工程師 Product/Test Engineer [限...,1,1,6001002015,新北市中和區,興南路一段142號,▋ 關於德州儀器\n德州儀器(TI)是位居類比晶片世界領導地位的半導體設計與製造公司，持續提...,...,"[外商公司, 員工2000人]",距捷運南勢角站320公尺,{'applyAnalyze': '//www.104.com.tw/jobs/apply/...,hotjob_chr,[2022科技菁英計畫] 產品測試工程師 Product/Test Engineer [限...,德州儀器工業股份有限公司,121.5083777,24.9871002,0,[工程學科類]
4,1,12781781,【外商】財務分析副理 Finance Planning &amp; Analysis Ass...,【外商】財務分析副理 Finance Planning &amp; Analysis Ass...,1,1,6001001004,台北市松山區,敦化北路88號13樓,Planning and Budgeting\n- Assist in the prepar...,...,"[外商公司, 員工250人]",距捷運台北小巨蛋站340公尺,{'applyAnalyze': '//www.104.com.tw/jobs/apply/...,hotjob_chr,【外商】財務分析副理 Finance Planning & Analysis Assista...,馬來西亞商白蘭氏三得利股份有限公司台灣分公司,121.5484798,25.0513338,0,"[財稅金融相關, 會計學相關, 企業管理相關]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1245,0,12916587,Scientist of Biology Group,<em class='b-txt--highlight'>Scientist</em> of...,1,1,6001001007,台北市信義區,基隆路一段333號20樓2013室,Insilico Medicine Taiwan is seeking to hire a ...,...,[員工13人],距捷運台北101/世貿站220公尺,{'applyAnalyze': '//www.104.com.tw/jobs/apply/...,jolist_c_relevance,Scientist of Biology Group,英科智能有限公司,121.5611792,25.0343865,0,[]
1246,0,12850348,微生物(資深) 研究員_Microbiology Senior(Sr.) Scientist,微生物(資深) 研究員_Microbiology Senior(Sr.) <em class...,1,1,6001002025,新北市五股區,五工六路25號,Position Summary:\nPharmacology Discovery Serv...,...,[員工130人],,{'applyAnalyze': '//www.104.com.tw/jobs/apply/...,jolist_c_relevance,微生物(資深) 研究員_Microbiology Senior(Sr.) Scientist,汎球生物科技股份有限公司,121.4485193,25.0627591,0,"[生物學相關, 醫學技術及檢驗相關, 藥學相關]"
1247,2,12823438,資料工程師 Data Engineer,資料工程師 <em class='b-txt--highlight'>Data</em> E...,1,1,6001001005,台北市大安區,近忠孝復興捷運站,與PM/UI/UX、前後端、資料、演算法工程師一同合作，開發與維護產品。\n 2. 負...,...,[員工650人],,{'applyAnalyze': '//www.104.com.tw/jobs/apply/...,jolist_c_relevance,資料工程師 Data Engineer,東森新媒體控股股份有限公司,121.5425991,25.0418146,0,[]
1248,0,13029724,Data Engineer 數據工程師,<em class='b-txt--highlight'>Data</em> Enginee...,1,1,6001016015,高雄市路竹區,,do we manage the complexities of our manufact...,...,[員工260人],,{'applyAnalyze': '//www.104.com.tw/jobs/apply/...,jolist_c_relevance,Data Engineer 數據工程師,德商默克在台集團_美商慧盛先進科技有限公司台灣分公司,120.2628111,22.8494042,0,"[自然科學學科類, 數學及電算機科學學科類]"


## Dump files for backup

* 因為抓取資料很久很辛苦，所以通常會把它寫到後端的檔案、資料庫，或者雲端的資料庫中。在此，我們選擇把他寫到pandas或者`.json()`比較好讀的json檔。通常不會寫到CSV，因為CSV是以逗點分隔為辨識基礎，但文本中可能也會有逗點，會比較容易出錯。
* 建議若設計了寫入，就立刻讀出來看看，省得不小心寫入錯誤，屆時程式關閉了，資料就取不回來了。
* 還有另外一種檔案是python的pickle檔。pickle是一種暫存檔，也就是我們現在如果有一個變數是pandas dataframe，寫到pickle再讀出來，他還會是一個pandas dataframe，而且資料型態都不會變。這點其實非常好用，因為如果你把一個dataframe轉為json，你要特別注意那些datetime欄位或multiindex的東西被轉成什麼，又是否能夠讀得回來。也就是如果時間物件要寫入到json檔時，理應要先轉成string，然後讀取回來要做分析時，也要再轉回來datetime。但如果你轉為pickle檔，存進去是什麼樣子，拿出來就是什麼樣子。那pickle檔有缺點嗎？有！就和其他的程式例如R或網頁程式無法通用，也無法直接寫到雲端資料庫。


### M1. Dump one variable to json by json library
* https://docs.python.org/3/library/json.html

In [23]:
import json
with open('104_list.json', 'w') as outfile:
    json.dump(all_data, outfile)

### M2. Dump and load json by pandas library

In [24]:
with open('104_df.json', 'w') as f:
    f.write(df.to_json())

In [25]:
with open("104_df.json") as fin:
    data2 = pd.read_json(fin)
data2.head()

,jobType,jobNo,jobName,jobNameSnippet,jobRole,jobRo,jobAddrNo,jobAddrNoDesc,jobAddress,description,...,tags,landmark,link,jobsource,jobNameRaw,custNameRaw,lon,lat,remoteWorkType,major
0,1,12901787,日班倉管員(週休二日免排班),日班倉管員(週休二日免排班),1,1,6001005012,桃園市大園區,,"1.倉儲貨品管理及貨品環境維護 \n2.物流倉儲各項改包進貨及出貨作業,熟悉倉庫生態佳 \n...",...,"[外商公司, 員工650人]",,{'applyAnalyze': '//www.104.com.tw/jobs/apply/...,hotjob_chr,日班倉管員(週休二日免排班),香港商台灣利豐物流有限公司台灣分公司_LF LOGISTICS (TAIWAN) LIMITED,121.193945,25.049263,0,[]
1,1,11726918,水晶軒專櫃店長(台中區)底薪固定13個月+獎金,水晶軒專櫃店長(台中區)底薪固定13個月+獎金,1,1,6001008004,台中市西區,,百貨公司專櫃儲備幹部及店長職務\r\n1.負責介紹及銷售商品。 \r\n2.提供顧客之接待與...,...,"[外商公司, 員工190人]",,{'applyAnalyze': '//www.104.com.tw/jobs/apply/...,hotjob_chr,水晶軒專櫃店長(台中區)底薪固定13個月+獎金,香港商施華洛世奇有限公司台灣分公司,120.663129,24.143060,0,[]
2,1,8422468,(Human Resources) Learning and Development Tea...,(Human Resources) Learning and Development Tea...,1,1,6001002011,新北市新店區,,About HTC\n\nHTC built a vision of the future ...,...,"[上市上櫃, 員工2000人]",,{'applyAnalyze': '//www.104.com.tw/jobs/apply/...,hotjob_chr,(Human Resources) Learning and Development Tea...,宏達電 HTC Corporation_宏達國際電子股份有限公司,121.539482,24.978282,0,"[心理學相關, 人力資源相關]"
3,1,12388507,[2022科技菁英計畫] 產品測試工程師 Product/Test Engineer [限...,[2022科技菁英計畫] 產品測試工程師 Product/Test Engineer [限...,1,1,6001002015,新北市中和區,興南路一段142號,▋ 關於德州儀器\n德州儀器(TI)是位居類比晶片世界領導地位的半導體設計與製造公司，持續提...,...,"[外商公司, 員工2000人]",距捷運南勢角站320公尺,{'applyAnalyze': '//www.104.com.tw/jobs/apply/...,hotjob_chr,[2022科技菁英計畫] 產品測試工程師 Product/Test Engineer [限...,德州儀器工業股份有限公司,121.508378,24.987100,0,[工程學科類]
4,1,12781781,【外商】財務分析副理 Finance Planning &amp; Analysis Ass...,【外商】財務分析副理 Finance Planning &amp; Analysis Ass...,1,1,6001001004,台北市松山區,敦化北路88號13樓,Planning and Budgeting\n- Assist in the prepar...,...,"[外商公司, 員工250人]",距捷運台北小巨蛋站340公尺,{'applyAnalyze': '//www.104.com.tw/jobs/apply/...,hotjob_chr,【外商】財務分析副理 Finance Planning & Analysis Assista...,馬來西亞商白蘭氏三得利股份有限公司台灣分公司,121.548480,25.051334,0,"[財稅金融相關, 會計學相關, 企業管理相關]"


### M3. Dump multiple variables to pickle

In [26]:
import pickle
with open('104.pkl', 'wb') as fout:  # Python 3: open(..., 'wb')
    pickle.dump([all_data, df], fout)

### Load multiple variables back to objects

In [27]:
with open('104.pkl', "rb") as fin:  # Python 3: open(..., 'rb')
    test = pickle.load(fin)
    print(type(test[0]))
    print(type(test[1]))

<class 'list'>
<class 'pandas.core.frame.DataFrame'>


In [28]:
test[0][0].keys()

dict_keys(['jobType', 'jobNo', 'jobName', 'jobNameSnippet', 'jobRole', 'jobRo', 'jobAddrNo', 'jobAddrNoDesc', 'jobAddress', 'description', 'optionEdu', 'period', 'periodDesc', 'applyCnt', 'applyDesc', 'custNo', 'custName', 'coIndustry', 'coIndustryDesc', 'salaryLow', 'salaryHigh', 'salaryDesc', 's10', 'appearDate', 'appearDateDesc', 'optionZone', 'isApply', 'applyDate', 'isSave', 'descSnippet', 'tags', 'landmark', 'link', 'jobsource', 'jobNameRaw', 'custNameRaw', 'lon', 'lat', 'remoteWorkType', 'major'])

## Using timestamp as file name

因為我們抓取資料的for-loop如果有幾千或幾萬個iterations，勢必很怕抓到一半斷掉或當機，卻又得重抓。所以最好的方式就是，每抓個幾百筆或幾千筆，就留一份檔案。但要如何區分這些檔案的先後順序？通常會用撈取的時間加上資料筆數的先後來作為檔名的一部分，這樣之後就可以觀看這些檔名來知道，究竟資料有沒有抓完，或者當在哪裡。

In [29]:
from datetime import datetime

now = datetime.now().strftime("%Y%m%d%H%M%S")
print("Current Time =", now)
with open('104_%s.pkl'%(now), 'wb') as fout:  # Python 3: open(..., 'wb')
    pickle.dump([all_data, df], fout)

Current Time = 20221010120556


## Clean version. 104


In [165]:
headers = {'referer': 'https://www.104.com.tw/'}
search_str = '數據分析'

# Detecting totalPage
url_104 = 'https://www.104.com.tw/jobs/search/list?ro=0&kwop=7&keyword=' + search_str + '&expansionType=area%2Cspec%2Ccom%2Cjob%2Cwf%2Cwktm&order=14&asc=0&page=2&mode=s&jobsource=2018indexpoc'
raw = requests.get(url_104, headers=headers).json()
totalPage = raw['data']['totalPage']
print(totalPage)

# Getting data by loop
all_data = []
for page in range(1, totalPage):
    url = 'https://www.104.com.tw/jobs/search/list?ro=0&kwop=7&keyword=' + search_str + '&expansionType=area%2Cspec%2Ccom%2Cjob%2Cwf%2Cwktm&order=14&asc=0&page=' + str(page) + '&mode=s&jobsource=2018indexpoc'
    raw = requests.get(url, headers=headers).json()
    all_data.extend(raw['data']['list'])
    print(len(all_data), end = " ")
print()
df = pd.DataFrame(all_data)


# Saving and Backing up data
import pickle
from datetime import datetime

now = datetime.now().strftime("%Y%m%d%H%M%S")
print("Current Time =", now)
with open('104_%s_%s.pkl'%(search_str, now), 'wb') as fout:  # Python 3: open(..., 'wb')
    pickle.dump([all_data, df], fout)

150
30 50 70 90 110 130 150 170 190 210 230 250 270 290 310 330 350 370 390 410 430 450 470 490 510 530 550 570 590 610 630 650 670 690 710 730 750 770 790 810 830 850 870 890 910 930 950 970 990 1010 1030 1050 1070 1090 1110 1130 1150 1170 1190 1210 1230 1250 1270 1290 1310 1330 1350 1370 1390 1410 1430 1450 1470 1490 1510 1530 1550 1570 1590 1610 1630 1650 1670 1690 1710 1730 1750 1770 1790 1810 1830 1850 1870 1890 1910 1930 1950 1970 1990 2010 2030 2050 2070 2090 2110 2130 2150 2170 2190 2210 2230 2250 2270 2290 2310 2330 2350 2370 2390 2410 2430 2450 2470 2490 2510 2530 2550 2570 2590 2610 2630 2650 2670 2690 2710 2730 2750 2770 2790 2810 2830 2850 2870 2890 2910 2930 2950 2970 2990 
Current Time = 20210325125357
